# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanoleo/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds a transparent baseline for prioritizing content-refresh opportunities.

The baseline uses only observed data from a fixed 90-day window ending 2026-06-30.

The rule is intentionally simple:

- Stale: `days_since_last_update >= 180`
- Visible: `impressions_90d >= 500`
- Score: observed `impressions_90d` for pages meeting both conditions

The output is decision-support, not a claim that a refresh will definitely improve performance.

In [74]:
# Install only what this notebook needs.

%pip -q install pandas numpy

In [75]:
import pandas as pd
import numpy as np

# Public anonymized starter dataset from this repository.
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Wanoleo/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print(f"Rows loaded: {len(df):,}")
print(f"Columns loaded: {len(df.columns):,}")

Rows loaded: 30,000
Columns loaded: 44


In [76]:
# Confirm the columns needed for ML-07 exist.

required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "days_since_last_update",
    "content_age_days",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("✓ All ML-07 required columns are present.")

✓ All ML-07 required columns are present.


## 1. My rule and its reason codes

### Rule

I will prioritize pages for refresh when they are both **stale** and **visible in search**.

- **Stale:** `days_since_last_update >= 180`
- **Visible:** `impressions_90d >= 500`

The score is the observed 90-day search impressions for pages that meet both conditions.

### Reason code

- `stale_visible_page` — the page is at least 180 days since its last update and has at least 500 observed search impressions.

### Action

- `refresh` — review the page for a possible content refresh.

This is a decision-support baseline, not a claim that every flagged page definitely needs updating.

In [77]:
# Basic preparation.
#
# These filters are NOT part of the scoring rule.
# They simply remove pages without observed search visibility
# and pages too new to be meaningful for this baseline.

work = df.copy()

work = work[
    (work["impressions_90d"] > 0) &
    (work["content_age_days"] >= 90)
].copy()

work = work.drop_duplicates(
    subset=["client_id", "content_id"]
).reset_index(drop=True)

print(f"Eligible pages: {len(work):,}")

Eligible pages: 30,000


### Signal 1 — Staleness

Staleness is directly linked to FlyRank's refresh flags.

I compare observed 90-day search impressions across freshness buckets.

The buckets are:

- `< 90 days`
- `90–179 days`
- `180–364 days`
- `365+ days`

The table reports the number of pages (`n`) and median observed impressions for each bucket.

Verdict: **[filled after running the code]**

The verdict is descriptive and directional; it does not claim that age causes search performance.

In [78]:
# Signal 1 — Staleness bucket table.
#
# This is linked to the real FlyRank refresh/staleness flag.

work["staleness_bucket"] = pd.cut(
    work["days_since_last_update"],
    bins=[-np.inf, 89, 179, 364, np.inf],
    labels=[
        "<90 days",
        "90–179 days",
        "180–364 days",
        "365+ days"
    ]
)

staleness_summary = (
    work
    .groupby(
        "staleness_bucket",
        observed=False
    )
    .agg(
        n=("content_id", "count"),
        median_impressions_90d=("impressions_90d", "median")
    )
    .reset_index()
)

print("SIGNAL 1 — STALENESS")
print("=" * 60)

display(staleness_summary)

SIGNAL 1 — STALENESS


,staleness_bucket,n,median_impressions_90d
0,<90 days,20655,472.0
1,90–179 days,9171,1692.0
2,180–364 days,169,16.0
3,365+ days,5,2.0


In [79]:
# One-word verdict for Signal 1.

stale_median = work.loc[
    work["days_since_last_update"] >= 180,
    "impressions_90d"
].median()

fresh_median = work.loc[
    work["days_since_last_update"] < 180,
    "impressions_90d"
].median()

if pd.isna(stale_median) or pd.isna(fresh_median):
    signal1_verdict = "FALSE"
elif stale_median > fresh_median:
    signal1_verdict = "CONFIRMED"
elif stale_median < fresh_median:
    signal1_verdict = "OPPOSITE"
else:
    signal1_verdict = "MIXED"

print(f"Signal 1 verdict: {signal1_verdict}")
print(f"Median impressions, stale: {stale_median:,.1f}")
print(f"Median impressions, non-stale: {fresh_median:,.1f}")

Signal 1 verdict: OPPOSITE
Median impressions, stale: 15.5
Median impressions, non-stale: 742.0


### Signal 2 — Search visibility

Search visibility is another signal used by FlyRank's rule-based flags.

I compare pages below and above the 500-impression threshold.

The table reports:

- `n`
- median observed 90-day impressions
- median days since last update

Verdict: **[filled after running the code]**

This is a directional check of whether the visibility threshold separates pages with meaningfully different observed search demand.

In [80]:
# Signal 2 — Search visibility bucket table.

work["visibility_bucket"] = np.where(
    work["impressions_90d"] >= 500,
    "500+ impressions",
    "Under 500 impressions"
)

visibility_summary = (
    work
    .groupby("visibility_bucket")
    .agg(
        n=("content_id", "count"),
        median_impressions_90d=("impressions_90d", "median"),
        median_days_since_last_update=("days_since_last_update", "median")
    )
    .reset_index()
)

print("SIGNAL 2 — SEARCH VISIBILITY")
print("=" * 60)

display(visibility_summary)

SIGNAL 2 — SEARCH VISIBILITY


,visibility_bucket,n,median_impressions_90d,median_days_since_last_update
0,500+ impressions,16726,2948.5,22.0
1,Under 500 impressions,13274,53.0,20.0


In [81]:
# One-word verdict for Signal 2.

high_visibility = work.loc[
    work["impressions_90d"] >= 500,
    "impressions_90d"
]

low_visibility = work.loc[
    work["impressions_90d"] < 500,
    "impressions_90d"
]

high_median = high_visibility.median()
low_median = low_visibility.median()

if pd.isna(high_median) or pd.isna(low_median):
    signal2_verdict = "FALSE"
elif high_median > low_median:
    signal2_verdict = "CONFIRMED"
elif high_median < low_median:
    signal2_verdict = "OPPOSITE"
else:
    signal2_verdict = "MIXED"

print(f"Signal 2 verdict: {signal2_verdict}")
print(f"Median impressions, 500+: {high_median:,.1f}")
print(f"Median impressions, under 500: {low_median:,.1f}")

Signal 2 verdict: CONFIRMED
Median impressions, 500+: 2,948.5
Median impressions, under 500: 53.0


## 2. Build the ranked queue

I use one transparent rule.

A page receives the `stale_visible_page` reason code when:

`days_since_last_update >= 180` **and** `impressions_90d >= 500`.

The score is:

`impressions_90d` for qualifying pages, otherwise `0`.

Therefore, the queue prioritizes pages that satisfy the refresh condition and have more observed search visibility.

The action is `refresh` for qualifying pages and `monitor` otherwise.

No product flag, future outcome, `trend_pct`, or `trend_direction` is used in the score.

In [82]:
# ML-07 baseline score.

STALE_DAYS = 180
VISIBLE_IMPRESSIONS = 500

stale = (
    work["days_since_last_update"] >= STALE_DAYS
).astype(int)

visible = (
    work["impressions_90d"] >= VISIBLE_IMPRESSIONS
).astype(int)

# Transparent score:
# stale * visible * observed impressions

work["baseline_score"] = (
    stale *
    visible *
    work["impressions_90d"]
)

work["reason_code"] = np.where(
    (stale == 1) & (visible == 1),
    "stale_visible_page",
    "not_selected"
)

work["action"] = np.where(
    work["reason_code"] == "stale_visible_page",
    "refresh",
    "monitor"
)

print(
    "Pages selected for refresh:",
    f"{(work['reason_code'] == 'stale_visible_page').sum():,}"
)

Pages selected for refresh: 17


In [83]:
# Rank the queue.

queue = (
    work
    .sort_values(
        by=[
            "baseline_score",
            "days_since_last_update",
            "impressions_90d"
        ],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

queue["baseline_rank"] = np.arange(
    1,
    len(queue) + 1
)

queue_output = queue[
    [
        "baseline_rank",
        "client_id",
        "content_id",
        "impressions_90d",
        "days_since_last_update",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

print("Ranked queue created.")
print(f"Total rows: {len(queue_output):,}")

Ranked queue created.
Total rows: 30,000


## 3. Top-10 review

I reviewed the top 10 pages selected by the baseline.

The baseline assigns `refresh` when a page is both stale and visible in search. The confidence note describes why the observed signals support the ranking, while the final column records what could make the recommendation wrong.

The review is intentionally skeptical: a high score means that a page meets the rule, not that a refresh is guaranteed to improve performance.

In [84]:
# Top-10 review.

top10 = (
    queue[
        queue["reason_code"] == "stale_visible_page"
    ]
    .head(10)
    .copy()
)

top10["why_its_here"] = (
    "Stale >= 180 days and visible >= 500 "
    "observed 90-day impressions."
)

top10["what_would_make_it_wrong"] = (
    "The fixed thresholds do not observe content quality, "
    "business priority, or whether a refresh would improve "
    "future performance."
)

top10_review = top10[
    [
        "baseline_rank",
        "client_id",
        "content_id",
        "impressions_90d",
        "days_since_last_update",
        "action",
        "why_its_here",
        "what_would_make_it_wrong"
    ]
].copy()

display(top10_review)

,baseline_rank,client_id,content_id,impressions_90d,days_since_last_update,action,why_its_here,what_would_make_it_wrong
0,1,client_7f2253d7e2,content_cf56e2e2e282,61678,194,refresh,Stale >= 180 days and visible >= 500 observed ...,The fixed thresholds do not observe content qu...
1,2,client_7f2253d7e2,content_7368877ea310,59472,194,refresh,Stale >= 180 days and visible >= 500 observed ...,The fixed thresholds do not observe content qu...
2,3,client_7f2253d7e2,content_1bfaa38ff26c,25715,194,refresh,Stale >= 180 days and visible >= 500 observed ...,The fixed thresholds do not observe content qu...
3,4,client_7f2253d7e2,content_0a91db491d14,13299,193,refresh,Stale >= 180 days and visible >= 500 observed ...,The fixed thresholds do not observe content qu...
4,5,client_7f2253d7e2,content_5feee3994adb,7812,194,refresh,Stale >= 180 days and visible >= 500 observed ...,The fixed thresholds do not observe content qu...
5,6,client_7f2253d7e2,content_c2d929d83eaa,7558,193,refresh,Stale >= 180 days and visible >= 500 observed ...,The fixed thresholds do not observe content qu...
6,7,client_7f2253d7e2,content_b16bd7307b39,4590,194,refresh,Stale >= 180 days and visible >= 500 observed ...,The fixed thresholds do not observe content qu...
7,8,client_7f2253d7e2,content_fe16a55cd13d,4556,194,refresh,Stale >= 180 days and visible >= 500 observed ...,The fixed thresholds do not observe content qu...
8,9,client_7f2253d7e2,content_ecb6215e79fd,4429,194,refresh,Stale >= 180 days and visible >= 500 observed ...,The fixed thresholds do not observe content qu...
9,10,client_7f2253d7e2,content_928af3e22c80,1697,193,refresh,Stale >= 180 days and visible >= 500 observed ...,The fixed thresholds do not observe content qu...


## 4. Weak picks + leakage check

### Weak picks

The weakest part of this baseline is its reliance on two fixed thresholds. A page just below 500 impressions is not selected even if it is very old, while a page just above 500 is selected. The rule also treats age as a proxy for refresh need and does not observe content quality or whether an update would actually improve performance.

Pages close to either threshold should therefore be treated as uncertain recommendations rather than ground truth.

### Leakage check

The baseline uses only observed content age, update age, and 90-day search-performance measurements.

I did not use:

- `trend_pct`
- `trend_direction`
- `is_declining_label`
- future-window outcomes
- FlyRank product flags or scores
- client names, URLs, or private queries

Therefore the score is intended as a transparent decision-support baseline rather than a reproduction of a product decision.

In [85]:
# Weak-pick analysis.

near_visibility = work[
    work["impressions_90d"].between(450, 550)
].copy()

near_staleness = work[
    work["days_since_last_update"].between(165, 195)
].copy()

print("WEAK-PICK CHECK")
print("=" * 60)

print(
    f"Pages within ±50 impressions of the 500 threshold: "
    f"{len(near_visibility):,}"
)

print(
    f"Pages within ±15 days of the 180-day threshold: "
    f"{len(near_staleness):,}"
)

print("\nClosest pages to the visibility threshold:")

display(
    near_visibility[
        [
            "client_id",
            "content_id",
            "impressions_90d",
            "days_since_last_update",
            "reason_code",
            "action"
        ]
    ]
    .assign(
        distance_to_threshold=lambda x:
        abs(x["impressions_90d"] - VISIBLE_IMPRESSIONS)
    )
    .sort_values("distance_to_threshold")
    .head(5)
)

print("\nClosest pages to the staleness threshold:")

display(
    near_staleness[
        [
            "client_id",
            "content_id",
            "impressions_90d",
            "days_since_last_update",
            "reason_code",
            "action"
        ]
    ]
    .assign(
        distance_to_threshold=lambda x:
        abs(x["days_since_last_update"] - STALE_DAYS)
    )
    .sort_values("distance_to_threshold")
    .head(5)
)

WEAK-PICK CHECK
Pages within ±50 impressions of the 500 threshold: 894
Pages within ±15 days of the 180-day threshold: 59

Closest pages to the visibility threshold:


,client_id,content_id,impressions_90d,days_since_last_update,reason_code,action,distance_to_threshold
571,client_19581e27de,content_983f6612f9ea,500,22,not_selected,monitor,0
3887,client_19581e27de,content_cd892ad205d3,500,22,not_selected,monitor,0
23598,client_19581e27de,content_09cea12992eb,500,22,not_selected,monitor,0
20643,client_f74efabef1,content_d61bf94dfdfb,500,8,not_selected,monitor,0
4831,client_349c41201b,content_ca56885db527,500,20,not_selected,monitor,0



Closest pages to the staleness threshold:


,client_id,content_id,impressions_90d,days_since_last_update,reason_code,action,distance_to_threshold
505,client_d029fa3a95,content_bfa3d6688324,27,183,not_selected,monitor,3
2653,client_d029fa3a95,content_1d10143d4e52,20,183,not_selected,monitor,3
3280,client_d029fa3a95,content_a34d943a132c,35,183,not_selected,monitor,3
3507,client_d029fa3a95,content_074ba6ead17b,533,183,stale_visible_page,refresh,3
3651,client_d029fa3a95,content_fd16e3475c29,429,183,not_selected,monitor,3


In [86]:
# Final leakage check.

score_features = {
    "days_since_last_update",
    "impressions_90d"
}

forbidden_features = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix",
    "is_quick_win"
]

print("LEAKAGE CHECK")
print("=" * 60)

print("Direct score inputs:")
for feature in sorted(score_features):
    print(f"  ✓ {feature}")

print("\nForbidden fields:")

for feature in forbidden_features:
    if feature in work.columns:
        print(f"  ✓ PRESENT BUT NOT USED: {feature}")
    else:
        print(f"  ✓ NOT PRESENT: {feature}")

# Verify the mathematical definition of the score.

expected_score = (
    (
        work["days_since_last_update"] >= STALE_DAYS
    ).astype(int)
    *
    (
        work["impressions_90d"] >= VISIBLE_IMPRESSIONS
    ).astype(int)
    *
    work["impressions_90d"]
)

assert np.array_equal(
    work["baseline_score"].to_numpy(),
    expected_score.to_numpy()
)

print("\n✓ Score uses only observed freshness and 90-day impressions.")
print("✓ No future outcome is used.")

LEAKAGE CHECK
Direct score inputs:
  ✓ days_since_last_update
  ✓ impressions_90d

Forbidden fields:
  ✓ PRESENT BUT NOT USED: trend_pct
  ✓ PRESENT BUT NOT USED: trend_direction
  ✓ NOT PRESENT: is_declining_label
  ✓ NOT PRESENT: health_score
  ✓ NOT PRESENT: priority_score
  ✓ NOT PRESENT: action_type
  ✓ NOT PRESENT: refresh_tier
  ✓ NOT PRESENT: needs_ctr_fix
  ✓ NOT PRESENT: is_quick_win

✓ Score uses only observed freshness and 90-day impressions.
✓ No future outcome is used.


In [87]:
# Final ML-07 sanity checks.

selected = work[
    work["reason_code"] == "stale_visible_page"
]

# At least ten are required for the requested top-10 review.
assert len(selected) >= 10, (
    f"Only {len(selected)} pages qualify. "
    "The assignment requires a top-10 review."
)

# Every selected page satisfies BOTH conditions.
assert (
    selected["days_since_last_update"] >= STALE_DAYS
).all()

assert (
    selected["impressions_90d"] >= VISIBLE_IMPRESSIONS
).all()

# Selected pages have the correct action.
assert (
    selected["action"] == "refresh"
).all()

# Non-selected pages have monitor.
non_selected = work[
    work["reason_code"] != "stale_visible_page"
]

assert (
    non_selected["action"] == "monitor"
).all()

# Top ten exists.
assert len(top10_review) == 10

print("FINAL SANITY CHECK")
print("=" * 60)
print("✓ At least 10 pages qualify")
print("✓ Every selected page satisfies both thresholds")
print("✓ Selected action = refresh")
print("✓ Non-selected action = monitor")
print("✓ Top-10 review contains 10 rows")
print("✓ Score definition verified")

FINAL SANITY CHECK
✓ At least 10 pages qualify
✓ Every selected page satisfies both thresholds
✓ Selected action = refresh
✓ Non-selected action = monitor
✓ Top-10 review contains 10 rows
✓ Score definition verified


In [88]:
# Write the required ML-07 output.

from pathlib import Path

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    OUTPUT_DIR /
    "baseline_action_score.csv"
)

queue_output.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"✓ Wrote: {OUTPUT_PATH}")
print(f"✓ Rows: {len(queue_output):,}")

✓ Wrote: work/outputs/baseline_action_score.csv
✓ Rows: 30,000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.